# insurance-telematics

**Raw telematics trip data to GLM-ready risk scores for UK motor insurance pricing.**

This notebook runs the full pipeline end-to-end in under two minutes:
simulate a synthetic fleet, classify driving behaviour with a Hidden Markov Model,
aggregate to driver-level risk scores, and produce a feature DataFrame
you can drop into your Poisson frequency GLM.

No raw telematics data required — `TripSimulator` generates a realistic
synthetic fleet of cautious, normal, and aggressive drivers with Poisson claims.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-telematics/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-telematics polars numpy

## Simulate a synthetic fleet

`TripSimulator` generates realistic 1Hz trip data for a mixed fleet.
Three driving types (cautious / normal / aggressive) drive Ornstein-Uhlenbeck
speed processes with different drift parameters.
Claims are Poisson with frequency proportional to time in the aggressive state.

In [ ]:
from insurance_telematics import TripSimulator

sim = TripSimulator(seed=42)
trips_df, claims_df = sim.simulate(
    n_drivers=200,
    trips_per_driver=30,   # 30 trips per driver observed
)

print(f"Trips:   {len(trips_df):,} rows")
print(f"Drivers: {trips_df['driver_id'].n_unique()} unique")
print(f"Claims:  {len(claims_df):,} claim events")
print(f"\nTrip columns: {trips_df.columns}")
trips_df.head(4)

## Run the full scoring pipeline

`TelematicsScoringPipeline` chains the four stages:
1. Clean trips (GPS jump removal, acceleration derivation)
2. Extract trip features (harsh braking rate, speeding fraction, night fraction)
3. Fit a 3-state HMM on the feature vectors (cautious / normal / aggressive latent states)
4. Aggregate to driver level using Bühlmann-Straub credibility weighting

The output is a driver-level DataFrame ready for a Poisson GLM.

In [ ]:
from insurance_telematics import TelematicsScoringPipeline

pipe = TelematicsScoringPipeline(n_hmm_states=3)
pipe.fit(trips_df, claims_df)

# Predict risk scores for all drivers
driver_scores = pipe.predict(trips_df)

print(f"Driver scores shape: {driver_scores.shape}")
print(f"Score columns: {driver_scores.columns}")
driver_scores.head(8)

## Validate: aggressive drivers should have higher risk scores

The simulator tags each driver with their true driving type.
If the pipeline is working, aggressive drivers (driving_type=2) should
have materially higher `telematics_risk_score` than cautious drivers (driving_type=0).

In [ ]:
import polars as pl

# Get true driver types from simulator
driver_types = sim.driver_metadata()  # polars DataFrame: driver_id, driving_type

# Join scores with true types
validation = (
    driver_scores
    .join(driver_types, on="driver_id", how="left")
    .group_by("driving_type")
    .agg([
        pl.col("telematics_risk_score").mean().alias("mean_risk_score"),
        pl.col("telematics_risk_score").std().alias("std_risk_score"),
        pl.len().alias("n_drivers"),
    ])
    .sort("driving_type")
)

type_labels = {0: "cautious", 1: "normal", 2: "aggressive"}
validation = validation.with_columns(
    pl.col("driving_type").replace(type_labels).alias("driver_type")
)
print(validation)

## Step-by-step pipeline (optional)

The `TelematicsScoringPipeline` is a convenience wrapper.
You can also run each stage independently for custom workflows.

In [ ]:
from insurance_telematics import (
    clean_trips,
    extract_trip_features,
    DrivingStateHMM,
    aggregate_to_driver,
)

# Stage 1: clean
trips_clean = clean_trips(trips_df)

# Stage 2: trip-level features
trip_features = extract_trip_features(trips_clean)
print(f"Trip features: {trip_features.columns}")

# Stage 3: HMM state classification
hmm = DrivingStateHMM(n_states=3)
hmm.fit(trip_features)
trip_features = hmm.predict_states(trip_features)
print(f"State distribution:\n{trip_features['hmm_state'].value_counts().sort('hmm_state')}")

# Stage 4: aggregate to driver level
driver_features = aggregate_to_driver(trip_features)
print(f"\nDriver-level features shape: {driver_features.shape}")
driver_features.head(4)

## What you should see

- Aggressive drivers should have a risk score 1.5–3x higher than cautious drivers.
  The exact ratio depends on the HMM fitting but the ordering should be consistent.
- The pipeline produces interpretable features (harsh_braking_rate, speeding_frac,
  pct_time_aggressive_state) that pricing actuaries can challenge.
- The step-by-step version shows the same result — the pipeline wrapper just chains them.

## Next steps

The full library also includes:

- **`ContinuousTimeHMM`** — continuous-time Markov chain for variable-length trip intervals
- **`load_trips()`** — loader for CSV or Parquet files from real telematics devices
- Integration with `insurance-credibility` for Bühlmann-Straub driver-level weighting

**GitHub:** https://github.com/burning-cost/insurance-telematics  
**PyPI:** https://pypi.org/project/insurance-telematics/